# 02 — Supplier Performance

## tl;dr

Across **1,800 PO lines**, on-time delivery is **83.4%**. Performance varies materially by supplier, and the highest spend does not automatically imply the highest operational risk. This notebook separates historical supplier performance from dependency and consequence.

## Context & Methods

This analysis asks: **How have suppliers performed over the observed 12-month purchasing period?** We summarize delivery, lateness, defects, lead-time consistency, and spend at the supplier grain.

### Key Assumptions

- A line is on time when receipt is on or before its promised date.
- Defect rate uses received quantity as the denominator.
- PO value equals ordered quantity multiplied by unit price.
- The analysis describes association and performance; it does not prove why a supplier was late or defective.

### 1. Load validated analytical outputs

In [1]:
from pathlib import Path
import pandas as pd

from src.supply_chain_risk.analytics import RISK_WEIGHTS, apply_weight_scenario, build_analysis, load_data

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
tables, supplier_scores, part_risk, kpis = build_analysis(DATA_DIR)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

supplier_scores.shape

(40, 28)

### 2. Review portfolio-level performance

In [2]:
pd.DataFrame([
    {"measure": "Supplier count", "value": len(supplier_scores)},
    {"measure": "PO-line count", "value": len(tables["purchase_orders"])},
    {"measure": "On-time delivery rate", "value": f"{kpis['on_time_delivery_rate']:.1%}"},
    {"measure": "Total PO value", "value": f"${supplier_scores['spend'].sum():,.0f}"},
])

,measure,value
0,Supplier count,40
1,PO-line count,1800
2,On-time delivery rate,83.4%
3,Total PO value,"$122,952,640"


### 3. Compare suppliers using consistent definitions

In [3]:
performance_columns = ["supplier_name", "country", "po_lines", "spend", "on_time_delivery_rate", "average_late_days", "defect_rate", "average_lead_time_days", "lead_time_cv"]
supplier_scores[performance_columns].sort_values(["on_time_delivery_rate", "defect_rate"], ascending=[True, False]).head(12).round(3)

,supplier_name,country,po_lines,spend,on_time_delivery_rate,average_late_days,defect_rate,average_lead_time_days,lead_time_cv
2,Vector Systems 20,India,38,"2,926,016.51",0.63,4.32,0.01,54.00,0.41
0,Aero Systems 10,Poland,39,"1,662,504.00",0.64,6.82,0.02,58.36,0.37
14,Nova Materials 21,Germany,21,"3,855,478.53",0.67,6.00,0.02,59.62,0.37
23,Pioneer Materials 33,Japan,9,"635,576.40",0.67,5.00,0.00,53.33,0.36
17,Vector Dynamics 01,Mexico,25,"2,392,266.66",0.68,4.16,0.02,54.80,0.25
4,Orion Dynamics 14,Canada,72,"4,952,504.40",0.72,5.29,0.02,57.56,0.42
8,Summit Systems 37,Poland,36,"1,482,455.76",0.72,3.42,0.02,55.53,0.37
12,Aero Precision 19,Canada,47,"2,914,981.63",0.72,3.75,0.01,59.87,0.36
9,Orion Precision 05,Mexico,76,"6,153,034.97",0.72,5.07,0.02,53.08,0.35
28,Titan Precision 38,Mexico,16,"728,256.91",0.75,4.69,0.01,59.50,0.30


### 4. Compare performance with commercial exposure

In [4]:
supplier_scores.nlargest(10, "spend")[["supplier_name", "spend", "on_time_delivery_rate", "defect_rate", "risk_score"]].round(3)

,supplier_name,spend,on_time_delivery_rate,defect_rate,risk_score
33,Aero Precision 08,"10,922,054.51",0.89,0.01,18.40
32,Orion Components 26,"8,260,869.75",0.94,0.01,18.60
25,Nova Dynamics 15,"6,528,609.73",0.76,0.00,22.70
9,Orion Precision 05,"6,153,034.97",0.72,0.02,32.10
30,Pioneer Components 09,"5,735,588.42",0.98,0.02,18.80
4,Orion Dynamics 14,"4,952,504.40",0.72,0.02,34.60
7,Nova Materials 32,"4,815,863.42",0.84,0.02,32.30
6,Atlas Precision 07,"4,448,808.40",0.85,0.02,32.40
38,Aero Components 36,"4,344,431.78",0.93,0.00,13.60
35,Pioneer Dynamics 18,"4,297,742.97",0.93,0.01,17.00


## Results

Performance measures reveal different supplier behaviors: some suppliers have weaker delivery reliability, others have higher quality losses, and high-spend suppliers are not uniformly the highest risk.

## Takeaways

- Review delivery, quality, lead-time stability, and spend together.
- Avoid ranking suppliers using one KPI alone.
- Investigate root causes before treating performance associations as causal.
- Use the next notebook to combine performance with dependency and consequence.